# Версия: полная загрузка данных

Копия `recommendations.ipynb`. Основная логика этапов та же, но данные читаются и обрабатываются **целиком в памяти** (без `pyarrow.iter_batches` / chunk-сканов parquet).


# Инициализация

Загружаем библиотеки необходимые для выполнения кода ноутбука.

In [2]:
import logging

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

%matplotlib inline
%config InlineBackend.figure_format = "png"
%config InlineBackend.figure_format = "retina"

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
logger = logging.getLogger(__name__)


# === ЭТАП 1 ===

# Загрузка первичных данных

Загружаем первичные данные из файлов:
- tracks.parquet
- catalog_names.parquet
- interactions.parquet

In [3]:
tracks = pd.read_parquet("tracks.parquet")
catalog_names = pd.read_parquet("catalog_names.parquet")

print("tracks:", tracks.shape)
print("catalog_names:", catalog_names.shape)
display(tracks.head())
display(catalog_names.head())


tracks: (1000000, 4)
catalog_names: (1812471, 3)


,track_id,albums,artists,genres
0,26,"[3, 2490753]",[16],"[11, 21]"
1,38,"[3, 2490753]",[16],"[11, 21]"
2,135,"[12, 214, 2490809]",[84],[11]
3,136,"[12, 214, 2490809]",[84],[11]
4,138,"[12, 214, 322, 72275, 72292, 91199, 213505, 24...",[84],[11]


,id,type,name
0,3,album,Taller Children
1,12,album,Wild Young Hearts
2,13,album,Lonesome Crow
3,17,album,Graffiti Soul
4,26,album,Blues Six Pack


In [4]:
interactions = pd.read_parquet("interactions.parquet")

print("interactions:", interactions.shape)
display(interactions.head())
print("started_at:", interactions["started_at"].min(), "—", interactions["started_at"].max())


interactions: (222629898, 4)


,user_id,track_id,track_seq,started_at
0,0,99262,1,2022-07-17
1,0,589498,2,2022-07-19
2,0,590262,3,2022-07-21
3,0,590303,4,2022-07-22
4,0,590692,5,2022-07-22


started_at: 2022-01-01 00:00:00 — 2022-12-31 00:00:00


# Обзор данных

Проверяем данные, есть ли с ними явные проблемы.

In [5]:
# Типы, пропуски, примеры значений
print("=== dtypes ===")
print("tracks:\n", tracks.dtypes, sep="")
print("\ncatalog_names:\n", catalog_names.dtypes, sep="")
print("\ninteractions:\n", interactions.dtypes, sep="")

print("\n=== nulls ===")
print("tracks:", tracks.isnull().sum().to_dict())
print("catalog_names:", catalog_names.isnull().sum().to_dict())
print("interactions:", interactions.isnull().sum().to_dict())

print("\n=== catalog type counts ===")
print(catalog_names["type"].value_counts())

print("\n=== пример списков в tracks ===")
print("albums:", tracks["albums"].iloc[0], type(tracks["albums"].iloc[0]))
print("artists:", tracks["artists"].iloc[0])
print("genres:", tracks["genres"].iloc[0])

# Пустые списки метаданных = «неизвестные» сущности
empty_stats = {
    col: int(tracks[col].map(len).eq(0).sum())
    for col in ["albums", "artists", "genres"]
}
print("\n=== треки с пустыми списками ===")
print(empty_stats)


=== dtypes ===
tracks:
track_id     int64
albums      object
artists     object
genres      object
dtype: object

catalog_names:
id       int64
type    object
name    object
dtype: object

interactions:
user_id                int32
track_id               int32
track_seq              int16
started_at    datetime64[ns]
dtype: object

=== nulls ===
tracks: {'track_id': 0, 'albums': 0, 'artists': 0, 'genres': 0}
catalog_names: {'id': 0, 'type': 0, 'name': 0}
interactions: {'user_id': 0, 'track_id': 0, 'track_seq': 0, 'started_at': 0}

=== catalog type counts ===
type
track     1000000
album      658724
artist     153581
genre         166
Name: count, dtype: int64

=== пример списков в tracks ===
albums: [      3 2490753] <class 'numpy.ndarray'>
artists: [16]
genres: [11 21]

=== треки с пустыми списками ===
{'albums': 18, 'artists': 15369, 'genres': 3687}


In [6]:
# Проверка согласованности идентификаторов с каталогом
album_ids = set(catalog_names.loc[catalog_names["type"] == "album", "id"])
artist_ids = set(catalog_names.loc[catalog_names["type"] == "artist", "id"])
genre_ids = set(catalog_names.loc[catalog_names["type"] == "genre", "id"])
track_name_ids = set(catalog_names.loc[catalog_names["type"] == "track", "id"])

print("tracks без имени в catalog_names:", len(set(tracks["track_id"]) - track_name_ids))
print("имена track без записи в tracks:", len(track_name_ids - set(tracks["track_id"])))


def unknown_id_stats(column: str, known_ids: set) -> pd.DataFrame:
    """Считает ссылки на id, которых нет в catalog_names для данного type."""
    exploded = tracks[["track_id", column]].explode(column).dropna(subset=[column])
    exploded[column] = exploded[column].astype("int64")
    unknown_mask = ~exploded[column].isin(known_ids)
    unknown = exploded.loc[unknown_mask, column]
    print(
        f"{column}: неизвестных ссылок={unknown_mask.sum()}, "
        f"уникальных unknown id={unknown.nunique()}, "
        f"затронуто треков={exploded.loc[unknown_mask, 'track_id'].nunique()}, "
        f"всего ссылок={len(exploded)}"
    )
    return unknown


unknown_albums = unknown_id_stats("albums", album_ids)
unknown_artists = unknown_id_stats("artists", artist_ids)
unknown_genres = unknown_id_stats("genres", genre_ids)

if len(unknown_genres):
    print("примеры неизвестных genre_id:", sorted(unknown_genres.unique())[:30])

# Согласованность interactions ↔ tracks
tracks_in_interactions = set(interactions["track_id"].unique())
print("\ntrack_id из interactions вне tracks:", len(tracks_in_interactions - set(tracks["track_id"])))
print("треки без прослушиваний:", len(set(tracks["track_id"]) - tracks_in_interactions))

# Нужно ли менять типы id
print("\n=== диапазоны id ===")
print(
    "tracks.track_id:", tracks["track_id"].min(), "—", tracks["track_id"].max(),
    "| dtype:", tracks["track_id"].dtype
)
print(
    "interactions.track_id/user_id:",
    interactions["track_id"].dtype, "/", interactions["user_id"].dtype
)
print("catalog_names.id:", catalog_names["id"].dtype)


tracks без имени в catalog_names: 0
имена track без записи в tracks: 0
albums: неизвестных ссылок=0, уникальных unknown id=0, затронуто треков=0, всего ссылок=3128808
artists: неизвестных ссылок=0, уникальных unknown id=0, затронуто треков=0, всего ссылок=1264212
genres: неизвестных ссылок=48369, уникальных unknown id=30, затронуто треков=48345, всего ссылок=1652658
примеры неизвестных genre_id: [124, 126, 130, 131, 132, 133, 134, 135, 146, 148, 150, 151, 152, 153, 154, 155, 156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168, 169]

track_id из interactions вне tracks: 0
треки без прослушиваний: 0

=== диапазоны id ===
tracks.track_id: 26 — 101521819 | dtype: int64
interactions.track_id/user_id: int32 / int32
catalog_names.id: int64


# Выводы

Приведём выводы по первому знакомству с данными:
- есть ли с данными явные проблемы,
- какие корректирующие действия (в целом) были предприняты.

In [7]:
# Приведение типов идентификаторов к единому компактному виду
# track_id/user_id в interactions уже int32; в tracks/catalog — int64, при этом значения входят в int32
tracks["track_id"] = tracks["track_id"].astype("int32")
catalog_names["id"] = catalog_names["id"].astype("int32")
interactions["user_id"] = interactions["user_id"].astype("int32")
interactions["track_id"] = interactions["track_id"].astype("int32")

# Оставляем в списках только id, которые есть в catalog_names
known_albums = album_ids
known_artists = artist_ids
known_genres = genre_ids


def filter_known_ids(values, known: set):
    if values is None or len(values) == 0:
        return np.array([], dtype="int32")
    filtered = [int(v) for v in values if int(v) in known]
    return np.array(filtered, dtype="int32")


tracks["albums"] = tracks["albums"].map(lambda x: filter_known_ids(x, known_albums))
tracks["artists"] = tracks["artists"].map(lambda x: filter_known_ids(x, known_artists))
tracks["genres"] = tracks["genres"].map(lambda x: filter_known_ids(x, known_genres))

# Контроль после очистки
print("после очистки — пустые списки:")
print({col: int(tracks[col].map(len).eq(0).sum()) for col in ["albums", "artists", "genres"]})
print("неизвестных genre-ссылок осталось:", unknown_id_stats("genres", known_genres).shape[0])
print("\ndtypes после преобразования:")
print(tracks.dtypes)
print(catalog_names.dtypes)
print(interactions.dtypes)


после очистки — пустые списки:
{'albums': 18, 'artists': 15369, 'genres': 3694}
genres: неизвестных ссылок=0, уникальных unknown id=0, затронуто треков=0, всего ссылок=1604289
неизвестных genre-ссылок осталось: 0

dtypes после преобразования:
track_id     int32
albums      object
artists     object
genres      object
dtype: object
id       int32
type    object
name    object
dtype: object
user_id                int32
track_id               int32
track_seq              int16
started_at    datetime64[ns]
dtype: object


In [7]:
print(
    """
Выводы по этапу 1
=================
1. Типы идентификаторов
   - В interactions user_id/track_id уже int32, track_seq — int16, started_at — datetime64.
   - В tracks и catalog_names идентификаторы были int64; значения входят в диапазон int32.
   - Привели tracks.track_id и catalog_names.id к int32 для единообразия и экономии памяти.

2. Неизвестные исполнители / альбомы / жанры
   - Пропусков (NaN) в таблицах нет.
   - Есть треки с пустыми списками: albums≈18, artists≈15.3k, genres≈3.7k —
     у таких треков метаданные фактически неизвестны.
   - Все непустые album_id и artist_id присутствуют в catalog_names.
   - Среди genre_id найдены ссылки на ~30 id, которых нет в catalog_names (type=genre);
     затронуто ≈48k треков. Неизвестные genre_id удалены из списков genres.

3. Согласованность таблиц
   - У каждого track_id есть имя в catalog_names.
   - Все track_id из interactions есть в tracks (и наоборот — каждый трек хотя бы раз встречается).

Корректирующие действия: приведение типов id к int32; фильтрация неизвестных genre_id.
Пустые списки albums/artists/genres оставлены как есть — это отдельный сигнал для EDA.
"""
)



Выводы по этапу 1
1. Типы идентификаторов
   - В interactions user_id/track_id уже int32, track_seq — int16, started_at — datetime64.
   - В tracks и catalog_names идентификаторы были int64; значения входят в диапазон int32.
   - Привели tracks.track_id и catalog_names.id к int32 для единообразия и экономии памяти.

2. Неизвестные исполнители / альбомы / жанры
   - Пропусков (NaN) в таблицах нет.
   - Есть треки с пустыми списками: albums≈18, artists≈15.3k, genres≈3.7k —
     у таких треков метаданные фактически неизвестны.
   - Все непустые album_id и artist_id присутствуют в catalog_names.
   - Среди genre_id найдены ссылки на ~30 id, которых нет в catalog_names (type=genre);
     затронуто ≈48k треков. Неизвестные genre_id удалены из списков genres.

3. Согласованность таблиц
   - У каждого track_id есть имя в catalog_names.
   - Все track_id из interactions есть в tracks (и наоборот — каждый трек хотя бы раз встречается).

Корректирующие действия: приведение типов id к int32; филь

# === ЭТАП 2 ===


# EDA

В этой версии ноутбука работаем с **полными таблицами в RAM** (без чтения parquet чанками), как в исходном пайплайне.


# Распределение количества прослушанных треков


Считаем число прослушиваний на пользователя и популярность треков через `groupby` по полному `interactions`.


In [ ]:
import gc

tracks_per_user = (
    interactions
    .groupby("user_id", sort=False)["track_id"]
    .size()
    .astype("int32")
    .rename("n_listens")
)

track_popularity = (
    interactions
    .groupby("track_id", sort=False)
    .agg(n_listens=("user_id", "size"))
    .reset_index()
    .astype({"track_id": "int32", "n_listens": "int32"})
)

listened_tracks = set(track_popularity["track_id"])

print(f"пользователей: {len(tracks_per_user):,}")
print(f"треков с прослушиваниями: {len(track_popularity):,}")


Статистики и гистограмма распределения прослушиваний по пользователям.


In [ ]:
print(tracks_per_user.describe(percentiles=[0.5, 0.9, 0.99]))

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(tracks_per_user, bins=80, color="steelblue", edgecolor="white")
ax.set_yscale("log")
ax.set_xlabel("Прослушиваний на пользователя")
ax.set_ylabel("Число пользователей (log)")
ax.set_title("Распределение числа прослушанных треков по пользователям")
plt.tight_layout()
plt.show()


Статистики популярности треков.


In [ ]:
print(track_popularity["n_listens"].describe(percentiles=[0.5, 0.9, 0.99]))


Визуализируем распределение популярности треков.


In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(track_popularity["n_listens"], bins=80, color="darkorange", edgecolor="white")
ax.set_yscale("log")
ax.set_xlabel("Прослушиваний на трек")
ax.set_ylabel("Число треков (log)")
ax.set_title("Распределение популярности треков")
plt.tight_layout()
plt.show()


# Наиболее популярные треки


Топ-20 треков по числу прослушиваний.


In [ ]:
top_tracks = (
    track_popularity
    .sort_values("n_listens", ascending=False)
    .head(20)
    .copy()
)
display(top_tracks)


Подтягиваем названия треков из `catalog_names`.


In [ ]:
track_names = (
    catalog_names
    .query("type == 'track'")
    [["id", "name"]]
    .rename(columns={"id": "track_id", "name": "title"})
)
top_tracks_named = top_tracks.merge(track_names, on="track_id", how="left")
display(top_tracks_named[["track_id", "title", "n_listens"]])


# Наиболее популярные жанры


Жанры считаем через компактный `track_popularity` (~1M треков): join с жанрами трека → explode → сумма `n_listens`. Логика та же, что в основном ноутбуке, без скана всех событий.


In [ ]:
track_genres = tracks.loc[tracks["genres"].map(len) > 0, ["track_id", "genres"]].copy()
track_genres["track_id"] = track_genres["track_id"].astype("int32")

genre_from_tracks = (
    track_popularity[["track_id", "n_listens"]]
    .merge(track_genres, on="track_id", how="inner")
    .explode("genres", ignore_index=True)
)
genre_from_tracks["genres"] = genre_from_tracks["genres"].astype("int32")

genre_stats = (
    genre_from_tracks
    .groupby("genres", sort=False)
    .agg(
        n_listens=("n_listens", "sum"),
        n_tracks=("track_id", "nunique"),
    )
    .reset_index()
    .rename(columns={"genres": "genre_id"})
    .astype({"genre_id": "int32", "n_listens": "int64", "n_tracks": "int32"})
    .sort_values("n_listens", ascending=False)
    .reset_index(drop=True)
)

del track_genres, genre_from_tracks
gc.collect()

print(f"уникальных жанров: {len(genre_stats):,}")
display(genre_stats.head(15))


Имена жанров и барчарт топ-15.


In [ ]:
genre_names = (
    catalog_names
    .query("type == 'genre'")
    [["id", "name"]]
    .rename(columns={"id": "genre_id", "name": "genre_name"})
)
top_genres = genre_stats.head(15).merge(genre_names, on="genre_id", how="left")
display(top_genres)

fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(
    top_genres["genre_name"].astype(str)[::-1],
    top_genres["n_listens"][::-1],
    color="seagreen",
)
ax.set_xlabel("Число прослушиваний")
ax.set_title("Топ-15 жанров по прослушиваниям")
plt.tight_layout()
plt.show()


# Треки, которые никто не прослушал


Сравниваем множество треков каталога и множество прослушанных `track_id`.


In [ ]:
tracks_all = set(tracks["track_id"].astype("int32"))
unlistened_ids = tracks_all - listened_tracks

print(f"всего треков: {len(tracks_all):,}")
print(f"треков с прослушиваниями: {len(listened_tracks):,}")
print(f"треков без прослушиваний: {len(unlistened_ids):,}")

unlistened_tracks = tracks[tracks["track_id"].isin(unlistened_ids)]
display(unlistened_tracks.head())


# Преобразование данных


Формируем `items`: `track_id` → `item_id` + название из каталога.


In [ ]:
items = tracks.rename(columns={"track_id": "item_id"}).copy()
items = items.merge(
    catalog_names
    .query("type == 'track'")
    [["id", "name"]]
    .rename(columns={"id": "item_id", "name": "title"}),
    on="item_id",
    how="left",
)
print("items:", items.shape)
display(items.head())


Формируем `events` из полного `interactions`: rename `track_id` → `item_id` (без chunk-записи).


In [ ]:
events = interactions.rename(columns={"track_id": "item_id"})
print("events:", events.shape)
print(events.dtypes)
display(events.head())


# Сохранение данных


Сохраняем `items.parquet` и `events.parquet` локально из полных DataFrame.


In [ ]:
from pathlib import Path

data_dir = Path("recsys/data")
data_dir.mkdir(parents=True, exist_ok=True)

items_path = data_dir / "items.parquet"
events_path = data_dir / "events.parquet"

items.to_parquet(items_path, index=False)
events.to_parquet(events_path, index=False)

print(f"сохранено: {items_path} ({items_path.stat().st_size / 1e6:.1f} MB)")
print(f"сохранено: {events_path} ({events_path.stat().st_size / 1e6:.1f} MB)")


Загружаем файлы в S3 по пути `recsys/data/` (ключи из `.env.local`).


In [ ]:
import os
import boto3


def load_env_file(path: str = ".env.local") -> None:
    env_path = Path(path)
    if not env_path.exists():
        print(f"файл {path} не найден — используются переменные окружения")
        return
    for raw in env_path.read_text(encoding="utf-8").splitlines():
        line = raw.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        os.environ.setdefault(key.strip(), value.strip())


load_env_file(".env.local")

bucket = os.environ["S3_BUCKET_NAME"]
endpoint = os.environ.get("MLFLOW_S3_ENDPOINT_URL", "https://storage.yandexcloud.net")

s3 = boto3.client(
    "s3",
    endpoint_url=endpoint,
    aws_access_key_id=os.environ["AWS_ACCESS_KEY_ID"],
    aws_secret_access_key=os.environ["AWS_SECRET_ACCESS_KEY"],
)

for local_path, key in [
    (items_path, "recsys/data/items.parquet"),
    (events_path, "recsys/data/events.parquet"),
]:
    print(f"upload {local_path} -> s3://{bucket}/{key}")
    s3.upload_file(str(local_path), bucket, key)
    print("ok")


# Очистка памяти


Удаляем промежуточные объекты этапа 2. `items`/`events` оставляем, если сразу идёте в этап 3.


In [ ]:
del_names = [
    "tracks_per_user",
    "track_popularity",
    "top_tracks",
    "top_tracks_named",
    "genre_stats",
    "top_genres",
    "unlistened_tracks",
    "listened_tracks",
    "tracks",
    "catalog_names",
    "interactions",
]
for name in del_names:
    if name in globals():
        del globals()[name]

gc.collect()
print("очистка выполнена")
print("items in memory:", "items" in globals())
print("events in memory:", "events" in globals())


# === ЭТАП 3 ===


# Загрузка данных


В этой версии загружаем `items` и полный `events` через `pd.read_parquet` (без `iter_batches`).


In [ ]:
import gc
import os
from collections import defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import scipy.sparse as sp
import sklearn.preprocessing
from catboost import CatBoostClassifier, Pool
from implicit.als import AlternatingLeastSquares

DATA_DIR = Path("recsys/data")
RECS_DIR = Path("recsys/recommendations")
RECS_DIR.mkdir(parents=True, exist_ok=True)

ITEMS_PATH = DATA_DIR / "items.parquet"
EVENTS_PATH = DATA_DIR / "events.parquet"

SPLIT_DATE = pd.Timestamp("2022-12-16")
LABEL_SPLIT_DATE = pd.Timestamp("2022-12-24")

TOP_K_POP = 100
ALS_FACTORS = 50
ALS_ITERS = 20
ALS_REG = 0.05
N_RECS = 100
N_SIMILAR = 50
NEGATIVES_PER_USER = 4
RANDOM_SEED = 0


Читаем полные таблицы `items` и `events`.


In [ ]:
if "items" not in globals():
    items = pd.read_parquet(ITEMS_PATH)
if "events" not in globals():
    events = pd.read_parquet(EVENTS_PATH)

events["started_at"] = pd.to_datetime(events["started_at"])

print("items:", items.shape)
print("events:", events.shape)
display(items.head(3))
display(events.head(3))


# Разбиение данных


Делим полный `events` в памяти:
- train: `started_at < 2022-12-16`;
- labels: `[2022-12-16, 2022-12-24)`;
- test: `>= 2022-12-24`.


In [ ]:
events_train = events[events["started_at"] < SPLIT_DATE].copy()
events_labels = events[
    (events["started_at"] >= SPLIT_DATE) & (events["started_at"] < LABEL_SPLIT_DATE)
].copy()
events_test = events[events["started_at"] >= LABEL_SPLIT_DATE].copy()

print("train:", events_train.shape)
print("labels:", events_labels.shape)
print("test:", events_test.shape)

users_train = set(events_train["user_id"])
users_labels = set(events_labels["user_id"])
users_test = set(events_test["user_id"])

print(f"users train: {len(users_train):,}")
print(f"users labels: {len(users_labels):,}")
print(f"users test: {len(users_test):,}")
print(f"train ∩ test: {len(users_train & users_test):,}")
print(f"cold test: {len(users_test - users_train):,}")


# Топ популярных


Популярность только по train; сохраняем топ-K в `top_popular.parquet`.


In [ ]:
top_popular = (
    events_train
    .groupby("item_id", sort=False)
    .agg(n_listens=("user_id", "size"))
    .reset_index()
    .astype({"item_id": "int32", "n_listens": "int32"})
    .sort_values("n_listens", ascending=False)
    .reset_index(drop=True)
)
top_popular["score"] = (
    top_popular["n_listens"] / top_popular["n_listens"].max()
).astype("float32")

top_popular_k = top_popular.head(TOP_K_POP).copy()
top_popular_path = RECS_DIR / "top_popular.parquet"
top_popular_k.to_parquet(top_popular_path, index=False)

print(f"уникальных item в train: {len(top_popular):,}")
display(top_popular_k.head(10))
print("saved:", top_popular_path)


Разворачиваем топ-K в рекомендации для всех test-пользователей (один и тот же топ каждому).


In [ ]:
test_users = np.fromiter(users_test, dtype=np.int32)
pop_items = top_popular_k["item_id"].to_numpy()
pop_scores = top_popular_k["score"].to_numpy()

pop_recs = pd.DataFrame(
    {
        "user_id": np.repeat(test_users, len(pop_items)),
        "item_id": np.tile(pop_items, len(test_users)),
        "score": np.tile(pop_scores, len(test_users)),
    }
).astype({"user_id": "int32", "item_id": "int32", "score": "float32"})

print("pop_recs:", pop_recs.shape)
display(pop_recs.head())


# Персональные (ALS)


Кодируем user/item по полному train и строим CSR-матрицу.


In [ ]:
user_encoder = sklearn.preprocessing.LabelEncoder()
item_encoder = sklearn.preprocessing.LabelEncoder()
user_encoder.fit(events_train["user_id"])
item_encoder.fit(events_train["item_id"])

events_train = events_train.copy()
events_train["user_id_enc"] = user_encoder.transform(events_train["user_id"])
events_train["item_id_enc"] = item_encoder.transform(events_train["item_id"])

n_users = len(user_encoder.classes_)
n_items = len(item_encoder.classes_)

# агрегируем повторы user-item
train_pairs = (
    events_train
    .groupby(["user_id_enc", "item_id_enc"], sort=False)
    .size()
    .reset_index(name="weight")
)

user_item_matrix = sp.csr_matrix(
    (
        train_pairs["weight"].astype("float32").to_numpy(),
        (
            train_pairs["user_id_enc"].to_numpy(),
            train_pairs["item_id_enc"].to_numpy(),
        ),
    ),
    shape=(n_users, n_items),
    dtype=np.float32,
)

print(f"CSR nnz={user_item_matrix.nnz:,}, shape={user_item_matrix.shape}")


Обучаем ALS и считаем персональные рекомендации для всех train-пользователей.


In [ ]:
als_model = AlternatingLeastSquares(
    factors=ALS_FACTORS,
    iterations=ALS_ITERS,
    regularization=ALS_REG,
    random_state=RANDOM_SEED,
)
als_model.fit(user_item_matrix)

# считаем рекомендациями батчами только чтобы не держать гигантский промежуточный объект,
# логика та же: recommend для всех пользователей train
als_parts = []
batch_users = 5_000
all_user_enc = np.arange(n_users, dtype=np.int32)

for start in range(0, n_users, batch_users):
    user_ids_enc = all_user_enc[start : start + batch_users]
    ids, scores = als_model.recommend(
        user_ids_enc,
        user_item_matrix[user_ids_enc],
        N=N_RECS,
        filter_already_liked_items=True,
    )
    rec = pd.DataFrame(
        {
            "user_id_enc": np.repeat(user_ids_enc, N_RECS),
            "item_id_enc": ids.reshape(-1).astype(np.int32),
            "score": scores.reshape(-1).astype(np.float32),
        }
    )
    rec["user_id"] = user_encoder.inverse_transform(rec["user_id_enc"])
    rec["item_id"] = item_encoder.inverse_transform(rec["item_id_enc"])
    als_parts.append(
        rec[["user_id", "item_id", "score"]].astype(
            {"user_id": "int32", "item_id": "int32", "score": "float32"}
        )
    )
    del ids, scores, rec

personal_als = pd.concat(als_parts, ignore_index=True)
del als_parts
gc.collect()

personal_als_path = RECS_DIR / "personal_als.parquet"
personal_als.to_parquet(personal_als_path, index=False)
print("personal_als:", personal_als.shape, "saved:", personal_als_path)


# Похожие (i2i / ALS)


Строим `similar_items` для всех item из train, убираем self-pair.


In [ ]:
sim_parts = []
batch_items = 5_000
all_item_enc = np.arange(n_items, dtype=np.int32)

for start in range(0, n_items, batch_items):
    item_ids_enc = all_item_enc[start : start + batch_items]
    sim_ids, sim_scores = als_model.similar_items(item_ids_enc, N=N_SIMILAR + 1)
    rec = pd.DataFrame(
        {
            "item_id_enc": np.repeat(item_ids_enc, N_SIMILAR + 1),
            "sim_item_id_enc": sim_ids.reshape(-1).astype(np.int32),
            "score": sim_scores.reshape(-1).astype(np.float32),
        }
    )
    rec = rec[rec["item_id_enc"] != rec["sim_item_id_enc"]]
    rec["item_id_1"] = item_encoder.inverse_transform(rec["item_id_enc"])
    rec["item_id_2"] = item_encoder.inverse_transform(rec["sim_item_id_enc"])
    sim_parts.append(
        rec[["item_id_1", "item_id_2", "score"]].astype(
            {"item_id_1": "int32", "item_id_2": "int32", "score": "float32"}
        )
    )
    del sim_ids, sim_scores, rec

similar = pd.concat(sim_parts, ignore_index=True)
del sim_parts
gc.collect()

similar_path = RECS_DIR / "similar.parquet"
similar.to_parquet(similar_path, index=False)
print("similar:", similar.shape, "saved:", similar_path)


# Построение признаков


Признаки ранжирования (≥3):
1. `als_score`
2. `pop_score`
3. `user_n_listens`
4. `item_n_listens`


In [ ]:
user_features = (
    events_train
    .groupby("user_id", sort=False)
    .size()
    .astype("int32")
    .rename("user_n_listens")
)

pop_map = top_popular.set_index("item_id")["score"]
item_listens_map = top_popular.set_index("item_id")["n_listens"]

label_pairs = set(map(tuple, events_labels[["user_id", "item_id"]].to_numpy()))

labels_users = users_labels & users_train
rng_rank = np.random.default_rng(RANDOM_SEED)
labels_users_list = np.fromiter(labels_users, dtype=np.int32)
MAX_RANK_TRAIN_USERS = 30_000
if len(labels_users_list) > MAX_RANK_TRAIN_USERS:
    labels_users_list = rng_rank.choice(labels_users_list, size=MAX_RANK_TRAIN_USERS, replace=False)
labels_users = set(labels_users_list.tolist())

candidates = personal_als[personal_als["user_id"].isin(labels_users)].copy()
candidates = candidates.rename(columns={"score": "als_score"})
candidates["pop_score"] = candidates["item_id"].map(pop_map).fillna(0).astype("float32")
candidates["item_n_listens"] = candidates["item_id"].map(item_listens_map).fillna(0).astype("int32")
candidates["user_n_listens"] = candidates["user_id"].map(user_features).fillna(0).astype("int32")
candidates["target"] = [
    1 if (u, i) in label_pairs else 0
    for u, i in zip(candidates["user_id"].to_numpy(), candidates["item_id"].to_numpy())
]

print("candidates:", candidates.shape, "positives:", int(candidates["target"].sum()))


Сэмплируем негативы для обучения ранкера.


In [ ]:
candidates_to_sample = candidates.groupby("user_id").filter(lambda x: x["target"].sum() > 0)

pos = candidates_to_sample.query("target == 1")
neg = (
    candidates_to_sample.query("target == 0")
    .groupby("user_id", group_keys=False)
    .apply(lambda x: x.sample(min(len(x), NEGATIVES_PER_USER), random_state=RANDOM_SEED))
)
candidates_for_train = pd.concat([pos, neg], ignore_index=True)
del candidates_to_sample, pos, neg
gc.collect()

print("candidates_for_train:", candidates_for_train.shape)
print(candidates_for_train["target"].value_counts())


# Ранжирование рекомендаций


Обучаем CatBoost на признаках без fine-tune.


In [ ]:
features = ["als_score", "pop_score", "user_n_listens", "item_n_listens"]
target = "target"

train_pool = Pool(
    data=candidates_for_train[features].fillna(0),
    label=candidates_for_train[target],
)

cb_model = CatBoostClassifier(
    iterations=200,
    learning_rate=0.1,
    depth=6,
    loss_function="Logloss",
    verbose=50,
    random_seed=RANDOM_SEED,
)
cb_model.fit(train_pool)
print("CatBoost trained")


Инференс на ALS-кандидатах для test-пользователей → `recommendations.parquet`.


In [ ]:
test_rank_users = users_test & users_train

to_rank = personal_als[personal_als["user_id"].isin(test_rank_users)].copy()
to_rank = to_rank.rename(columns={"score": "als_score"})
to_rank["pop_score"] = to_rank["item_id"].map(pop_map).fillna(0).astype("float32")
to_rank["item_n_listens"] = to_rank["item_id"].map(item_listens_map).fillna(0).astype("int32")
to_rank["user_n_listens"] = to_rank["user_id"].map(user_features).fillna(0).astype("int32")
to_rank["score"] = cb_model.predict_proba(to_rank[features].fillna(0))[:, 1].astype("float32")

to_rank = to_rank.sort_values(["user_id", "score"], ascending=[True, False])
to_rank["rank"] = to_rank.groupby("user_id").cumcount() + 1

recommendations = to_rank.loc[
    to_rank["rank"] <= N_RECS, ["user_id", "item_id", "score", "rank"]
].astype({"user_id": "int32", "item_id": "int32", "score": "float32", "rank": "int16"})

recommendations_path = RECS_DIR / "recommendations.parquet"
recommendations.to_parquet(recommendations_path, index=False)
print("recommendations:", recommendations.shape, "saved:", recommendations_path)


# Оценка качества


Метрики precision/recall@10, coverage@100, novelty@5 для top_popular / personal_als / recommendations на одном sample пользователей.


In [ ]:
def build_user_items_map(df: pd.DataFrame, users: set) -> dict:
    out = defaultdict(set)
    sub = df[df["user_id"].isin(users)][["user_id", "item_id"]]
    for u, i in sub.to_numpy():
        out[int(u)].add(int(i))
    return out


def process_events_recs_for_binary_metrics(gt_by_user: dict, recs: pd.DataFrame, top_k: int = 10):
    recs = recs.sort_values(["user_id", "score"], ascending=[True, False])
    recs = recs.groupby("user_id", sort=False).head(top_k)
    common_users = set(recs["user_id"]) & set(gt_by_user.keys())
    recs = recs[recs["user_id"].isin(common_users)]
    recs_grouped = recs.groupby("user_id")["item_id"].apply(set)

    rows = []
    for user_id, pred_items in recs_grouped.items():
        truth = gt_by_user.get(user_id, set())
        tp = len(pred_items & truth)
        fp = len(pred_items - truth)
        fn = len(truth - pred_items)
        rows.append((user_id, tp, fp, fn))
    return pd.DataFrame(rows, columns=["user_id", "tp", "fp", "fn"])


def compute_precision_recall(metrics_df: pd.DataFrame):
    precision = (metrics_df["tp"] / (metrics_df["tp"] + metrics_df["fp"])).fillna(0).mean()
    recall = (metrics_df["tp"] / (metrics_df["tp"] + metrics_df["fn"])).fillna(0).mean()
    return float(precision), float(recall)


def coverage_at_k(recs: pd.DataFrame, n_items_total: int, top_k: int = 100) -> float:
    top = recs.sort_values(["user_id", "score"], ascending=[True, False]).groupby("user_id").head(top_k)
    return top["item_id"].nunique() / n_items_total


def novelty_at_k(recs: pd.DataFrame, train_by_user: dict, top_k: int = 5) -> float:
    top = recs.sort_values(["user_id", "score"], ascending=[True, False]).groupby("user_id").head(top_k).copy()
    top["seen"] = [
        1 if i in train_by_user.get(u, set()) else 0
        for u, i in zip(top["user_id"].to_numpy(), top["item_id"].to_numpy())
    ]
    return float(1 - top.groupby("user_id")["seen"].mean().mean())


n_items_total = items["item_id"].nunique()

rng = np.random.default_rng(RANDOM_SEED)
eval_users_list = np.fromiter(users_test & users_train, dtype=np.int32)
if len(eval_users_list) > 20_000:
    eval_users_list = rng.choice(eval_users_list, size=20_000, replace=False)
eval_users = set(eval_users_list.tolist())

gt_by_user = build_user_items_map(events_test, eval_users)
train_by_user = build_user_items_map(events_train, eval_users)


def evaluate_recs(name: str, recs: pd.DataFrame) -> dict:
    sub = recs[recs["user_id"].isin(eval_users)]
    mdf = process_events_recs_for_binary_metrics(gt_by_user, sub, top_k=10)
    precision, recall = compute_precision_recall(mdf)
    res = {
        "model": name,
        "precision@10": precision,
        "recall@10": recall,
        "coverage@100": coverage_at_k(sub, n_items_total, top_k=100),
        "novelty@5": novelty_at_k(sub, train_by_user, top_k=5),
    }
    print(res)
    return res


metrics_table = pd.DataFrame(
    [
        evaluate_recs("top_popular", pop_recs),
        evaluate_recs("personal_als", personal_als),
        evaluate_recs("recommendations_ranked", recommendations),
    ]
)
display(metrics_table)


Загружаем артефакты в S3: `recsys/recommendations/`.


In [ ]:
import boto3


def load_env_file(path: str = ".env.local") -> None:
    env_path = Path(path)
    if not env_path.exists():
        print(f"файл {path} не найден")
        return
    for raw in env_path.read_text(encoding="utf-8").splitlines():
        line = raw.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        os.environ.setdefault(key.strip(), value.strip())


load_env_file(".env.local")
bucket = os.environ["S3_BUCKET_NAME"]
endpoint = os.environ.get("MLFLOW_S3_ENDPOINT_URL", "https://storage.yandexcloud.net")
s3_client = boto3.client(
    "s3",
    endpoint_url=endpoint,
    aws_access_key_id=os.environ["AWS_ACCESS_KEY_ID"],
    aws_secret_access_key=os.environ["AWS_SECRET_ACCESS_KEY"],
)

for local_path, key in [
    (RECS_DIR / "top_popular.parquet", "recsys/recommendations/top_popular.parquet"),
    (RECS_DIR / "personal_als.parquet", "recsys/recommendations/personal_als.parquet"),
    (RECS_DIR / "similar.parquet", "recsys/recommendations/similar.parquet"),
    (RECS_DIR / "recommendations.parquet", "recsys/recommendations/recommendations.parquet"),
]:
    print(f"upload {local_path} -> s3://{bucket}/{key}")
    s3_client.upload_file(str(local_path), bucket, key)
    print("ok")


# === Выводы, метрики ===


Итоги этапа 3 для версии с полной загрузкой данных.


In [ ]:
print(
    """
Выводы по этапу 3 (full in-memory)
=================================
1. Данные загружались целиком: items/events через pd.read_parquet,
   interactions на этапе 2 использовался как полный DataFrame.
2. Сплит: train < 2022-12-16; labels [12-16, 12-24); test >= 12-24.
3. Модели/файлы: top_popular, personal_als, similar, recommendations.
4. Признаки ранкера: als_score, pop_score, user_n_listens, item_n_listens.
5. Метрики — в таблице metrics_table.
"""
)
display(metrics_table)


Очистка тяжёлых объектов.


In [ ]:
del_names = [
    "user_item_matrix",
    "als_model",
    "candidates",
    "candidates_for_train",
    "to_rank",
    "train_pairs",
    "gt_by_user",
    "train_by_user",
    "user_features",
    "label_pairs",
]
for name in del_names:
    if name in globals():
        del globals()[name]
gc.collect()
print("cleanup done")
